<a href="https://colab.research.google.com/github/bzeot/colab_inclass/blob/main/Hocsauproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
# Cài đặt công cụ điều khiển Roboflow
!pip install roboflow
!pip install roboflow torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.0/184.0 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 120.2 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


In [4]:
import os
from roboflow import Roboflow
rf = Roboflow(api_key="b37R5x0iKmGwvFdQT2PI")
project = rf.workspace("projectai-5qavw").project("phan-loai-dien-thoai")

base_path = '/content/drive/MyDrive/AI_Project'

# Duyệt qua các thư mục Asus, Iphone, Samsung...
for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)
    if os.path.isdir(folder_path):
        print(f"Đang tải dữ liệu lớp: {folder}")
        for img in os.listdir(folder_path):
            project.upload(os.path.join(folder_path, img), num_retry_uploads=3)

loading Roboflow workspace...
loading Roboflow project...
Đang tải dữ liệu lớp: Tablet
Đang tải dữ liệu lớp: Xiaomi
Đang tải dữ liệu lớp: Iphone
Đang tải dữ liệu lớp: Laptop
Đang tải dữ liệu lớp: Samsung
Đang tải dữ liệu lớp: Asus
Đang tải dữ liệu lớp: Realme


In [6]:
dataset = project.version(1).download("pytorch")

RuntimeError: Version number 1 is not found.

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models

# 1. Tải bộ dữ liệu đã được gán nhãn và tăng cường (Augmented)
# dataset = project.version(1).download("pytorch") # Thay bằng code Export của bạn

# 2. Tiền xử lý để AI tập trung vào Logo và Camera
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder('/content/phan-loai-dien-thoai-1/train', transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)

# 3. Khởi tạo mô hình ResNet50
model = models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 7) # 7 lớp đầu ra

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# 4. Chạy 50 Epoch
for epoch in range(50):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/50 - Loss: {total_loss/len(train_loader):.4f}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/phan-loai-dien-thoai-1/train'